In [1]:
import numpy as np
import torch
from tqdm import tqdm
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.parallel import DistributedDataParallel, DataParallel
from utils import get_sigma_time, get_sample_time, VESDE, get_config
from model import UNet3DModel
import matplotlib.pyplot as plt
from torch_ema import ExponentialMovingAverage
import logging
import os
import sys
from os.path import join
import argparse

In [2]:
from dataclasses import dataclass

@dataclass
class args:
    config = 'configs/config_camels.json'
    disable_tqdm = False

In [3]:
config_filename = args.config
enable_tqdm = not args.disable_tqdm
config = get_config(config_filename)

In [4]:
Nside = config.data.image_size
#DEVICE = config.device
DEVICE = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')


sigma_time = get_sigma_time(config.model.sigma_min, config.model.sigma_max)
sample_time = get_sample_time(config.model.sampling_eps, config.model.T)

cosmo_dir = config.model.cosmo_dir
data_path = join(config.model.workdir, cosmo_dir)
checkpoint_dir = join(data_path, config.model.checkpoint_dir)

In [6]:
def get_filepath(sample_no, file_type):
    if file_type == 'z0':
        return f"Train_z0_2000/{sample_no}_z0.npy"
    elif file_type == 'z127':
        return f"Train_z127_from_IC_2000/df_m_z=127_sim{sample_no}.npy"
    elif file_type == 'halo':
        return f"halo_LH_128/halo_lh_{sample_no:04d}.npy"
    elif file_type == 'recon':
        return f"Recon_z127_2000/{sample_no}.npy"
    elif file_type == 'latent_z0':
        return f"Latent_z0/{sample_no}.npy"
    elif file_type == 'latent_z127':
        return f"Latent_z127/{sample_no}.npy"
    elif file_type == 'camels_z0':
        return f"Train_z0_CAMELS/z0_{sample_no:04d}.npy"
    elif file_type == 'camels_z127':
        return f"Train_z127_CAMELS/z127_{sample_no:04d}.npy"
    else:
        raise ValueError(f"Unknown file type: {file_type}")

In [ ]:
os.path.exists()

In [13]:
data_root = '../IC-Flow-Diffusion/Dataset/'
sample_no = 998

sample_dir = os.path.join(data_path, 'samples', str(sample_no))
if not os.path.exists(sample_dir):
    os.makedirs(sample_dir, exist_ok=True)

z127_path = os.path.join(data_root, get_filepath(sample_no, 'camels_z127'))
z0_path = os.path.join(data_root, get_filepath(sample_no, 'camels_z0'))

In [14]:
N = config.data.image_size
z0 = np.load(z0_path).reshape(N, N, N)
noise_sigma = config.data.noise_sigma
z0_noisy = z0 + noise_sigma * np.random.normal(size=z0.shape)
z0_noisy = z0_noisy[np.newaxis, ...]  # shape: (1, 128, 128, 128)

# === Load z=127 and normalize ===
z127 = np.load(z127_path).reshape(N, N, N)
z127_norm = (z127 - np.mean(z127)) / np.std(z127)
z127_norm = z127_norm[np.newaxis, ...]

# === Save as observation and truth ===
np.save(os.path.join(sample_dir, "observation.npy"), z0_noisy)
np.save(os.path.join(sample_dir, "truth.npy"), z127_norm)

print(f"✅ Saved observation and truth to {sample_dir}")

✅ Saved observation and truth to run/cosmos_camels/samples/998


In [16]:
# Build pytorch dataloaders
input_data = np.float32(np.load(join(sample_dir, 'observation.npy')))
print("Loaded shape:", input_data.shape)
label_data = np.float32(np.load(join(sample_dir, 'truth.npy')))
input_data = torch.from_numpy(input_data).to(DEVICE)
label_data = torch.from_numpy(label_data).to(DEVICE)
input_data = torch.unsqueeze(input_data, dim=1)
label_data = torch.unsqueeze(label_data, dim=1)

Loaded shape: (1, 128, 128, 128)


In [17]:
# Initialize score model
model = UNet3DModel(config)
#model = DataParallel(model)
model = model.to(DEVICE)

ema = ExponentialMovingAverage(model.parameters(), decay=config.model.ema_rate)

sde = VESDE(config.model.sigma_min, config.model.sigma_max, config.model.num_scales, config.model.T, config.model.sampling_eps)

In [ ]:
# Check for existing checkpoint
checkpoint_path = join(checkpoint_dir, 'checkpoint.pth')
if os.path.isfile(checkpoint_path):
    loaded_state = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(loaded_state['model'], strict=False)
    ema.load_state_dict(loaded_state['ema'])
    logging.info(f"Loaded checkpoint from {checkpoint_path}.")
    print(f"Loaded checkpoint from {checkpoint_path}.")
else:
    logging.warning(f"No checkpoint found at {checkpoint_path}. Starting from scratch.")
    print(f"No checkpoint found at {checkpoint_path}. Starting from scratch.")

model = model.eval()

Loaded checkpoint from run/cosmos_camels/checkpoints/checkpoint.pth.


UNet3DModel(
  (act): SiLU()
  (all_modules): ModuleList(
    (0): GaussianFourierProjection()
    (1): Linear(in_features=64, out_features=128, bias=True)
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): Conv3d(2, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (4-5): 2 x ResnetBlockBigGANpp(
      (GroupNorm_0): GroupNorm(8, 32, eps=1e-06, affine=True)
      (Conv_0): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (Dense_0): Linear(in_features=128, out_features=32, bias=True)
      (GroupNorm_1): GroupNorm(8, 32, eps=1e-06, affine=True)
      (Dropout_0): Dropout(p=0.1, inplace=False)
      (Conv_1): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (act): SiLU()
    )
    (6): ResnetBlockBigGANpp(
      (GroupNorm_0): GroupNorm(8, 32, eps=1e-06, affine=True)
      (Conv_0): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (Dense_0): Linear(in_fea

In [19]:
def one_step(x, t):
    t_vec = torch.ones(shape[0], device=DEVICE) * t
    model_output = model(torch.cat([x, input_data], dim=1), t_vec)
    x, x_mean = sde.update_fn(x, t_vec, model_output=model_output)
    return x, x_mean

print("input_data shape before tiling:", input_data.shape)


input_data = torch.tile(input_data, dims=(config.sampling.batch_size, 1, 1, 1, 1))
shape = (config.sampling.batch_size, 1, Nside, Nside, Nside)

input_data shape before tiling: torch.Size([1, 1, 128, 128, 128])


In [ ]:
samples = []
print('Sampling begins.')
for j in tqdm(
    # range(config.sampling.num_samples//config.sampling.batch_size),
    range(2),
    disable=args.disable_tqdm
):
    with torch.no_grad(), ema.average_parameters():
        x = sde.prior_sampling(shape).to(DEVICE)
        timesteps = sde.timesteps.to(DEVICE)
        for i in tqdm(range(sde.N), disable=args.disable_tqdm):
            t = timesteps[i]
            x, x_mean = one_step(x, t)
        samples.append(x_mean.detach().cpu().numpy())
    np.save( os.path.join(sample_dir, 'sample.npy'), np.array(samples))
    print(f'Finished {j+1}th round')

print('Done sampling')
np.save(os.path.join(sample_dir, 'sample.npy'), np.array(samples).squeeze())

Sampling begins.


 50%|█████     | 1/2 [04:11<04:11, 251.84s/it]

Finished 1th round


100%|██████████| 2/2 [08:22<00:00, 251.22s/it]

Finished 2th round
Done sampling


In [21]:
np.save(os.path.join(sample_dir, 'sample.npy'), np.array(samples).squeeze())